In [1]:
from pathlib import Path
import shutil
import re
import pandas as pd

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()
Path.mkdir(Path("examenes_quimica"), exist_ok=True)
EXAMS_PATH = Path("../examenes_quimica")

In [3]:
def get_files(path = EXAMS_PATH):
    return sorted(path.rglob("*.pdf"))

In [4]:
# def clasificar_examen(file: Path) -> str | None:

#     folder = file.parent.name
#     start_position = file.name.find(folder) +1
#     text_to_scan = file.name[start_position:].lower()

#     parts_added = []
#     parts = {"extra" : "extraordinaria",
#              "ord" : "ordinaria",
#              "titular" : "titular",
#              "suplente" : "suplente",
#              "reserva a" : "reserva A",
#              "reserva b" : "reserva B",
#              "junio": "junio",
#              "sept": "septiembre"}
    
#     for k, v in parts.items():
#         if k in text_to_scan:
#             parts_added.append(v)

#     if parts_added:
#         return folder + "_" + "_".join(parts_added) 
#     else:
#         return file.name

Extraer archivos de subcarpetas

In [5]:
for folder in EXAMS_PATH.iterdir():
    if folder.is_dir():
        # find pdf files
        pdf_files = [f for f in folder.rglob("*.pdf")]

        for f in pdf_files:

            # get subfolders of pdf files
            subfolder = str(f.relative_to(folder).parent).replace("/", " - ")

            # move them into folder root
            dest = folder/ f"{subfolder} - {f.name}"

            # move the file to folder base
            shutil.move(f, dest)
            
        # remove subfolders
        for dir in folder.iterdir():
                if dir.is_dir():
                    shutil.rmtree(dir)

Borrar archivos de criterios

In [6]:
for file in get_files():
    new_file = file.name.lower()

    # remove file if it is a criteria file
    if "criterio" in file.name.lower() or "_c_" in file.name.lower():
        file.unlink(missing_ok=True)
        continue

## Limpiar nombres de archivo

Convertir a minúsculas, eliminar espacios extra y convertir guiones y espacios a guión bajo.

In [7]:
old_filenames_df = pd.DataFrame(get_files(), index = range(len(get_files())), columns = ["old_filename"] )

In [8]:
old_filenames_df

,old_filename
0,../examenes_quimica/2010_quimica/quimica - qui...
1,../examenes_quimica/2010_quimica/quimica - qui...
2,../examenes_quimica/2010_quimica/quimica - qui...
3,../examenes_quimica/2010_quimica/quimica - qui...
4,../examenes_quimica/2010_quimica/quimica - qui...
...,...
104,../examenes_quimica/2026_quimica/Quimica - Ext...
105,../examenes_quimica/2026_quimica/Quimica - Ext...
106,../examenes_quimica/2026_quimica/Quimica - Ord...
107,../examenes_quimica/2026_quimica/Quimica - Ord...


In [9]:
for file in get_files():
    filename = file.name
    # eliminar espacios extra
    filename = re.sub(r"\s{2,}", " ", filename)
    filename = (filename
                .replace(" ", "_")
                .replace("-", "_")
                .lower()
    )

    # tras la sustitución, eliminar guiones bajos duplicados
    filename = re.sub(r"_{2,}", "_", filename)

    # corregir variantes de química
    variants = ["química", "quimica", "qu╓mica", "quínica"]
    pattern = "|".join(map(re.escape, variants))
    filename = re.sub(pattern, "química", filename)

    # eliminar la palabra química
    filename = filename.replace("química", "")

    # reemplazar estas strings por ""
    strings_to_clean = ['exámenes', "cabe", "andalucía", "anda", "examen", "impreso", "corregido", "rev", "20_21"]
    pattern = "|".join(map(re.escape, strings_to_clean))
    filename = re.sub(pattern, "", filename)

    # eliminar números al final del nombre
    filename = re.sub(r"_\d+.pdf$", ".pdf", filename)

    # eliminar sel_2011
    filename = re.sub(r"^sel_\d+", "", filename)

    # eliminar guiones bajo al principio y final
    filename = filename[:-4].strip("_") + ".pdf"

    # renombrar extra, ord a junio, julio, septiembre
    year = int(file.parts[-2][:4])

    if year <= 2020:
            filename = (filename
                            .replace("extra_", "septiembre_")
                            .replace("extraordinaria_", "septiembre_")
                            .replace("ordinaria_", "junio_")
                            .replace("ord_", "junio_")
                        
            )
    else:
        filename = (filename
                        .replace("extra_", "julio_")
                        .replace("extraordinaria_", "julio_")
                        .replace("ordinaria_", "junio_")
                        .replace("ord_", "junio_")     
        )

    # eliminar "ua" (universidad almería)
    filename = filename.replace("ua", "")
    
    # eliminar palabras duplicadas
    filename_words = filename.split("_")
    unique_filename_words = []
    for w in filename_words:
        if w not in unique_filename_words:
            unique_filename_words.append(w)
    
    filename = "_".join(unique_filename_words)

    # añadir año y asignatura al nombre de archivo
    filename = f"{str(year)}_quimica_{filename}"


    new_file = f"{file.parent}/{filename}"
    shutil.move(file, new_file)

In [18]:
new_filenames_df = pd.DataFrame(get_files(), index = range(len(get_files())), columns = ["new_filename"] )

filenames_df = pd.concat([old_filenames_df, new_filenames_df], axis=1)

for col in filenames_df.columns:
    filenames_df[col] = filenames_df[col].replace(r"^\.\./examenes_quimica/", "", regex=True)

In [21]:
filenames_df.to_csv(ROOT / "filenames.csv", index = False)

In [107]:
for folder in sorted((ROOT / "examenes_quimica").iterdir()):
    if folder.is_dir():
        files_count = len(list(folder.iterdir()))
        if  files_count != 6:
            print(f"Error en carpeta {folder}. {files_count} archivos.")

Error en carpeta /home/daniel/code/selector_examenes/examenes_quimica/2012_quimica. 14 archivos.
Error en carpeta /home/daniel/code/selector_examenes/examenes_quimica/2016_quimica. 5 archivos.


- 2012 tiene las opciones por separado (opciones A y B). Además, en dos de los exámenes están las opciones por separado.y un pdf con las dos opciones juntas. Requiere manipulación de los ejercicios.
  
- 2016 tiene solo 5 ficheros. Falta un examen.